<div dir="rtl">

# دسته‌بندی گزارش‌های حوادث دریایی MAIB: اجرای کامل پروژه

این نوت‌بوک گام‌های «روش تحقیق» پروپوزال را به ترتیب اجرا می‌کند. هر سلول فقط یک فایل `stepN_*.py` را اجرا می‌کند.
کد اصلی داخل همان فایل‌هاست و هر فایل در ابتدایش توضیح می‌دهد چه کاری انجام می‌دهد.

| سلول | فایل | گام پروپوزال |
|---|---|---|
| ۱ | `step1_prepare_data.py` | گام ۱: دادگان، پاک‌سازی، تقسیم ۷۰/۱۵/۱۵ |
| ۲ | `step2_tfidf_svm.py` | گام ۲-الف: خط مبنای TF-IDF + SVM |
| ۳ | `step4_run_experiments.py` (که `step3_train.py` را صدا می‌زند) | گام‌های ۲-ب، ۲-ج، ۳ و ۴ |
| ۴ | `step5_report.py` | گام ۵: جدول‌ها و ماتریس درهم‌ریختگی |
| ۵ | `step6_attention.py` | گام ۵: تحلیل کیفی وزن‌های توجه |

**کجا اجرا کنیم؟** آموزش BERT روی لپ‌تاپ (GTX 1650 Ti) حدود ۲۰ ساعت طول می‌کشد. روی GPU رایگان Kaggle (T4) حدود ۳ تا ۴ ساعت.
این نوت‌بوک بدون هیچ تغییری روی هر دو کار می‌کند.

</div>

<div dir="rtl">

## ۰. آماده‌سازی

</div>

In [ ]:
import os
from IPython.display import HTML, Image, display

import torch

ON_KAGGLE = os.path.exists("/kaggle/working")
if ON_KAGGLE:                                    # on Kaggle: check the settings, then download the project
    assert torch.cuda.is_available(), (
        "No GPU. Right panel -> Session options -> Accelerator -> GPU T4 (needs a verified phone number).")
    if not os.path.exists("/kaggle/working/AI-Paper"):
        !git clone -q -b claude/awesome-newton-43m2x0 https://github.com/The-Dark-Lord2002/AI-Paper.git /kaggle/working/AI-Paper
    assert os.path.exists("/kaggle/working/AI-Paper"), (
        "Could not download the code. Right panel -> Session options -> Internet -> On (needs a verified phone number).")
    %cd /kaggle/working/AI-Paper/maib-thesis
    !pip install -q -r requirements.txt

BATCH_SIZE = 32 if ON_KAGGLE else 8              # Kaggle T4 has 16 GB memory, the laptop GPU has 4 GB
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - training will be very slow")

<div dir="rtl">

## ۱. دادگان (گام ۱ پروپوزال)
بارگذاری ۵٬۷۶۸ گزارش، پاک‌سازی، حذف تکراری‌ها، حذف کلاس‌های کمتر از ۲۰ گزارش و تقسیم stratified.

</div>

In [ ]:
!python step1_prepare_data.py
display(Image("reports/class_distribution.png"))

<div dir="rtl">

## ۲. خط مبنای کلاسیک: TF-IDF + SVM (گام ۲-الف)
چند ثانیه روی CPU. عددی که BERT باید از آن بهتر شود.

</div>

In [ ]:
!python step2_tfidf_svm.py

<div dir="rtl">

## ۳. آموزش همه‌ی مدل‌های BERT (گام‌های ۲-ب، ۲-ج، ۳ و ۴)
۶ پیکربندی × ۳ seed = **۱۸ اجرا**. فهرست پیکربندی‌ها در `config.py` و متغیر `EXPERIMENTS` است.

اگر اجرا قطع شد، همین سلول را دوباره اجرا کن. اجراهای تمام‌شده رد می‌شوند.

</div>

In [ ]:
!python step4_run_experiments.py --batch-size $BATCH_SIZE

<div dir="rtl">

## ۴. نتایج (گام ۵)
جدول اصلی (میانگین ± انحراف معیار روی ۳ seed)، جواب دو سؤال پژوهش، recall هر کلاس و ماتریس درهم‌ریختگی.

</div>

In [ ]:
!python step5_report.py
display(Image("reports/per_class_recall.png"))
display(Image("reports/confusion_matrix.png"))

<div dir="rtl">

## ۵. تحلیل وزن‌های توجه (گام ۵، تحلیل کیفی)
برای هر کلاس، نمونه‌های درست و نادرست با رنگ‌آمیزی کلمات بر اساس وزن توجه.

</div>

In [ ]:
!python step6_attention.py
display(HTML(open("reports/attention_examples.html").read()))

<div dir="rtl">

## ۶. فقط در Kaggle: بسته‌بندی خروجی‌ها
بعد از اجرا، فایل `outputs.zip` را از تب **Output** دانلود کن و داخل پوشه‌ی `maib-thesis` لپ‌تاپ باز کن.

</div>

In [ ]:
if ON_KAGGLE:
    !zip -qr /kaggle/working/outputs.zip results reports
    print("Download /kaggle/working/outputs.zip from the Output tab")